# Module 20 — Multi-Agent Architectures

Build a supervisor/worker runtime with typed task envelopes, capability isolation, bounded fan-out, deadlines, idempotency and result validation.

In [ ]:
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
import time, uuid


In [ ]:
@dataclass(frozen=True)
class Task:
    id:str; parent:str|None; tenant:str; worker:str; caps:frozenset; payload:dict; deadline:float; idem:str

@dataclass(frozen=True)
class Result:
    id:str; worker:str; ok:bool; evidence:list; confidence:float; error:str|None=None

class Worker:
    name='base'; caps=frozenset()
    def run(self,t): raise NotImplementedError


In [ ]:
class Research(Worker):
    name='research'; caps=frozenset({'kb:read'})
    def run(self,t): return Result(t.id,self.name,True,['source evidence'],.9)

class Finance(Worker):
    name='finance'; caps=frozenset({'finance:write'})
    def run(self,t): return Result(t.id,self.name,True,['financial evidence'],.8)

workers={w.name:w for w in [Research(),Finance()]}
completed=set()

def dispatch(t):
    if t.id in completed: return Result(t.id,t.worker,True,['deduplicated'],1)
    if time.monotonic()>=t.deadline: return Result(t.id,t.worker,False,[],0,'deadline')
    w=workers[t.worker]
    if not t.caps.issubset(w.caps): raise PermissionError('capability contract violation')
    result=w.run(t)
    if result.ok: completed.add(t.id)
    return result

def task(worker,caps,payload,tenant='t1',timeout=5):
    return Task(str(uuid.uuid4()),None,tenant,worker,frozenset(caps),payload,time.monotonic()+timeout,str(uuid.uuid4()))


## 1. Fan-out/fan-in
Create three independent tasks and run them with bounded concurrency.

In [ ]:
tasks=[task('research',{'kb:read'},{'query':q}) for q in ['finance','legal','technical']]
with ThreadPoolExecutor(max_workers=2) as pool:
    results=list(pool.map(dispatch,tasks))
results


## 2. Capability escalation
Try assigning `finance:write` to the research worker. The runtime must reject it before execution.

In [ ]:
try:
    dispatch(task('research',{'finance:write'},{}))
except PermissionError as e:
    print('DENIED:',e)


## 3. Duplicate delivery
Dispatch the same task twice and prove the second delivery does not execute a side effect again.

In [ ]:
t=task('research',{'kb:read'},{}); print(dispatch(t)); print(dispatch(t))


## 4. Contract validation
A successful worker result must contain identity, evidence and confidence in [0,1]. Extend the validator to enforce schema versions and provenance.

In [ ]:
def validate(r):
    if not r.id or not r.worker: raise ValueError('missing identity')
    if r.ok and not r.evidence: raise ValueError('successful result needs evidence')
    if not 0<=r.confidence<=1: raise ValueError('bad confidence')
    return True

for r in results: print(validate(r))


## 5. Failure injection
Exercises: crash one worker, expire a deadline, duplicate a task, return malformed output, request unauthorized capability, and simulate recursive delegation. Add recovery and regression tests for every case.

## 6. Production extensions
1. Replace in-memory completion with the Module 1 durable store.
2. Add cancellation tokens.
3. Add global and per-worker budgets.
4. Add queue-based dispatch.
5. Add dead-letter tasks.
6. Add independent verification.
7. Add correlation IDs and structured traces.
8. Integrate Module 18 capability and approval policy.
9. Benchmark sequential vs bounded parallel execution.
10. Build a 50-case regression suite.

# Gold challenge
Build AegisAI Supervisor/Worker Runtime: durable task envelopes → capability-scoped workers → bounded fan-out/fan-in → retries/cancellation → partial-failure recovery → verification → graph-level audit. Demonstrate measurable improvement over the Module 19 baseline.